# Weekly OpenRouter Usage

This notebook focuses on long-history weekly token usage, provider share shifts, and the programming category slice.


In [3]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd

BASE_DIR = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").exists())
SRC_DIR = BASE_DIR / "src"
for path in (BASE_DIR, SRC_DIR):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
print(f"Base dir: {BASE_DIR}")


Base dir: /Users/henrywzh/Desktop/Quant/alternative-data


In [4]:
from research_data.api import build_weekly_openrouter_usage, weekly_tokens

weekly_usage = build_weekly_openrouter_usage(base_dir=BASE_DIR, refresh=False)
print("Rows:", len(weekly_usage))
weekly_usage.head()


Rows: 1550


,week_start_date,time_grain,dataset_source,entity_type,entity_id,entity_name,parent_entity_id,parent_entity_name,metric_name,metric_unit,metric_value,rank,category_slug,source_url,source_run_id,scraped_at
0,2025-03-30,week,market_share,author,google,google,<NA>,<NA>,token_share_pct,share,8.249661e+10,1,<NA>,https://openrouter.ai/rankings,20260404T120606Z-ef7072ee,2026-04-04T12:06:06.720967Z
1,2025-03-30,week,market_share,author,anthropic,anthropic,<NA>,<NA>,token_share_pct,share,5.479196e+10,2,<NA>,https://openrouter.ai/rankings,20260404T120606Z-ef7072ee,2026-04-04T12:06:06.720967Z
2,2025-03-30,week,market_share,author,openai,openai,<NA>,<NA>,token_share_pct,share,5.035977e+10,3,<NA>,https://openrouter.ai/rankings,20260404T120606Z-ef7072ee,2026-04-04T12:06:06.720967Z
3,2025-03-30,week,market_share,author,deepseek,deepseek,<NA>,<NA>,token_share_pct,share,3.776712e+10,4,<NA>,https://openrouter.ai/rankings,20260404T120606Z-ef7072ee,2026-04-04T12:06:06.720967Z
4,2025-03-30,week,market_share,author,openrouter,openrouter,<NA>,<NA>,token_share_pct,share,2.477639e+10,5,<NA>,https://openrouter.ai/rankings,20260404T120606Z-ef7072ee,2026-04-04T12:06:06.720967Z


## Weekly Totals By Dataset Source


In [5]:
weekly_totals = (
    weekly_usage.groupby(["week_start_date", "dataset_source"], as_index=False)["metric_value"]
    .sum()
    .sort_values(["week_start_date", "dataset_source"])
)
print(weekly_totals.tail(12).to_string(index=False))
weekly_totals.tail(12)


week_start_date         dataset_source  metric_value
     2026-03-16 categories_programming  7.196714e+12
     2026-03-16             top_models  2.035031e+13
     2026-03-22           market_share  2.238398e+13
     2026-03-23 categories_programming  9.786983e+12
     2026-03-23             top_models  2.272984e+13
     2026-03-29           market_share  2.439982e+13
     2026-03-30 categories_programming  1.478517e+13
     2026-03-30             top_models  2.703850e+13
     2026-04-05           market_share  4.712052e+12
     2026-04-06 categories_programming  8.133213e+12
     2026-04-06             top_models  2.103569e+13
     2026-04-12           market_share  3.299536e+12


,week_start_date,dataset_source,metric_value
143,2026-03-16,categories_programming,7.196714e+12
144,2026-03-16,top_models,2.035031e+13
145,2026-03-22,market_share,2.238398e+13
146,2026-03-23,categories_programming,9.786983e+12
147,2026-03-23,top_models,2.272984e+13
148,2026-03-29,market_share,2.439982e+13
149,2026-03-30,categories_programming,1.478517e+13
150,2026-03-30,top_models,2.703850e+13
151,2026-04-05,market_share,4.712052e+12
152,2026-04-06,categories_programming,8.133213e+12


## Example Filters

The helpers below show how to grab just the models or providers you care about.


In [6]:
selected_models = weekly_tokens(
    models=["openai/gpt-4.1", "anthropic/claude-sonnet-4", "google/gemini-2.5-pro"],
    base_dir=BASE_DIR,
    refresh=False,
)
selected_authors = weekly_tokens(
    authors=["openai", "anthropic", "google"],
    base_dir=BASE_DIR,
    refresh=False,
)
print(selected_models.tail(10).to_string(index=False))
print(selected_authors.tail(10).to_string(index=False))


week_start_date time_grain dataset_source entity_type entity_id entity_name parent_entity_id parent_entity_name     metric_name metric_unit  metric_value  rank category_slug                     source_url             source_run_id                  scraped_at
     2026-04-12       week   market_share      author    google      google             <NA>               <NA> token_share_pct       share  6.736449e+11     1          <NA> https://openrouter.ai/rankings 20260413T085855Z-8d64bd20 2026-04-13T08:58:55.317410Z
     2026-04-12       week   market_share      author anthropic   anthropic             <NA>               <NA> token_share_pct       share  4.887642e+11     2          <NA> https://openrouter.ai/rankings 20260413T085855Z-8d64bd20 2026-04-13T08:58:55.317410Z
     2026-04-12       week   market_share      author   minimax     minimax             <NA>               <NA> token_share_pct       share  3.849037e+11     3          <NA> https://openrouter.ai/rankings 20260413T085855Z-8

In [7]:
programming_slice = weekly_usage[weekly_usage["dataset_source"] == "categories_programming"].copy()
programming_leaders = (
    programming_slice.sort_values(["week_start_date", "rank"]) 
    .groupby("week_start_date", as_index=False)
    .first()[["week_start_date", "entity_id", "metric_value"]]
)
programming_leaders.tail(10)


,week_start_date,entity_id,metric_value
37,2026-02-02,moonshotai/kimi-k2.5-0127,1.333577e+12
38,2026-02-09,minimax/minimax-m2.5-20260211,2.022037e+12
39,2026-02-16,minimax/minimax-m2.5-20260211,3.004803e+12
40,2026-02-23,minimax/minimax-m2.5-20260211,1.358292e+12
41,2026-03-02,minimax/minimax-m2.5-20260211,1.739039e+12
42,2026-03-09,minimax/minimax-m2.5-20260211,1.604357e+12
43,2026-03-16,Others,2.089819e+12
44,2026-03-23,xiaomi/mimo-v2-pro-20260318,3.284141e+12
45,2026-03-30,qwen/qwen3.6-plus-04-02:free,4.712438e+12
46,2026-04-06,Others,2.313746e+12
